# Part 2 Test-Time Scaling

This notebook loads an AIME 2024 dataset, runs a model on each problem, extracts an AIME-style final answer, and grades the outputs.

In [1]:
import re

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
MODEL_NAME = "Qwen/Qwen3-4B"
# or allenai/Olmo-3-7B-Thinking
DATASET_NAME = "OpenRLHF/aime-2024"
MAX_NEW_TOKENS = 4096 #512

## Loading the model and the data

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(DATASET_NAME, split="train")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Evaluation helpers

In [4]:
def extract_thinking_trace(text):
    match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    return match.group(1).strip() if match else ""


def strip_thinking_trace(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|begin_of_thought\|>.*?<\|end_of_thought\|>", "", text, flags=re.DOTALL)
    return text.strip()



def extract_answer(text: str, mode="exact_match") -> int | None:
    """Extract an AIME-style integer answer from a model completion."""
    answer_text = strip_thinking_trace(text)
    if not answer_text:
        if mode == "exact_match":
            return None
        else:
            answer_text = text  # fall back to full text


    # 1. Boxed LaTeX answer: \boxed{123}
    if mode == "exact_match":
        boxed = re.findall(r"\\boxed\{(\d+)\}", answer_text)
        if boxed:
            val = int(boxed[-1])
            return val
        else:
            return None

    elif mode == "flexible_extract":
        # 2. "The answer is N" or "answer: N" patterns
        patterns = [
            r"(?:the\s+)?answer\s+is\s+[:\s]*(\d+)",
            r"answer[:\s]+(\d+)",
            r"=\s*(\d+)\s*$",
            r"(?:therefore|thus|so),?\s+(\d+)\s*(?:\.|$)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, answer_text, re.IGNORECASE)
            if matches:
                val = int(matches[-1])
                return val

        # 3. Last integer in [0, 999] in the answer portion
        integers = re.findall(r"\b(\d{1,3})\b", answer_text)
        for candidate in reversed(integers):
            val = int(candidate)
            return val
        return None


## 2.1 Warm-Up

You can also explore using vLLM to speed up inference!

In [ ]:
ENABLE_THINKING = True  # Ensure thinking is enabled for this analysis
ANSWER_MODE = "flexible_extract"

model.to(device)
model.eval()

records = []

for i, example in enumerate(dataset):
    problem = example["prompt"][0]["content"]
    gold_answer = int(example["label"])

    messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                {"role": "user", "content": problem}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    # Decode only the newly generated tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    model_output = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # Debug: print raw tags on first example to confirm tag format
    if i == 0:
        print("RAW OUTPUT SNIPPET:", repr(model_output[:300]))

    extracted = extract_answer(model_output, mode=ANSWER_MODE)
    if extracted is not None:
        correct = extracted == gold_answer
    else:
        correct = False

    # Extract thinking trace and calculate its token length
    extracted_thinking = extract_thinking_trace(model_output)
    thinking_tokens = tokenizer.encode(extracted_thinking, add_special_tokens=False)
    thinking_length = len(thinking_tokens)

    records.append({
        "problem": problem,
        "gold_answer": gold_answer,
        "model_output": model_output,
        "extracted_answer": extracted,
        "correct": correct,
        "thinking_length": thinking_length, # Add thinking length to records
    })

    print(f"[{i+1}/{len(dataset)}] gold={gold_answer} pred={extracted} correct={correct} thinking_len={thinking_length}")
    print(model_output[-200:])

results_df = pd.DataFrame(records)

print("\nDistribution of Thinking Lengths (in tokens):")
print(results_df["thinking_length"].describe())

results_df

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
results_df["correct"].mean()

In [ ]:
ENABLE_THINKING = False  # Set False for no-thinking condition
ANSWER_MODE = "exact_match"

model.to(device)
model.eval()

records = []

for i, example in enumerate(dataset):
    problem = example["prompt"]
    gold_answer = int(example["label"])

    messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                {"role": "user", "content": problem}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    # Decode only the newly generated tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    model_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

    extracted = extract_answer(model_output, mode=ANSWER_MODE)
    if extracted is not None:
        correct = extracted == gold_answer
    else:
        correct = False

    records.append({
        "problem": problem,
        "gold_answer": gold_answer,
        "model_output": model_output,
        "extracted_answer": extracted,
        "correct": correct,
    })

    print(f"[{i+1}/{len(dataset)}] gold={gold_answer} pred={extracted} correct={correct}")

results_df = pd.DataFrame(records)
results_df